# [실습 05] 생성자↔비평자 2-에이전트 협업

> **연계**: 제3부 05장(다중 에이전트) · **환경**: Google Colab · **모델**: 오픈웨이트 `Qwen/Qwen2.5-1.5B-Instruct`

**학습 목표**
- **역할 분담**(생성자·비평자)으로 품질이 오르는 것을 체험한다(05-1).
- 생성→비평→수정의 **상호 견제** 루프를 구현한다.

In [ ]:
!pip install -q transformers accelerate torch

In [ ]:
import torch
from transformers import pipeline
gen = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
               torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
               device_map="auto")

def ask(system, user):
    msg = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    return gen(msg, max_new_tokens=200, do_sample=False)[0]["generated_text"][-1]["content"].strip()

## 1. 두 역할(프로파일) 정의

In [ ]:
WRITER = "너는 카피라이터다. 요청받은 홍보 문구를 한 문장으로 쓴다."
CRITIC = "너는 깐깐한 편집자다. 문구의 약점을 1가지만 지적하고 개선 방향을 제시한다."

## 2. 생성 → 비평 → 수정 루프

In [ ]:
topic = "친환경 텀블러"
draft = ask(WRITER, f"{topic} 홍보 문구를 써줘.")
print("[생성자 초안]", draft)

critique = ask(CRITIC, f"다음 문구를 비평해줘: {draft}")
print("[비평자 지적]", critique)

final = ask(WRITER, f"원래 문구: {draft}\n편집자 의견: {critique}\n의견을 반영해 한 문장으로 다시 써줘.")
print("[생성자 수정본]", final)

## 3. 정리
- 한 모델이 **역할을 바꿔** 생성자·비평자로 협업했다(관심사 분리, 05-1).
- 생성자 단독보다 비평자의 견제가 품질을 높인다.
- **더 해보기**: 비평→수정을 2회 반복하고 매 라운드 결과를 비교해 보세요.